### 下載檔案

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("joebeachcapital/30000-spotify-songs")

print("Path to dataset files:", path)

Path to dataset files: /Users/angus3938/.cache/kagglehub/datasets/joebeachcapital/30000-spotify-songs/versions/2


In [3]:
import pandas as pd

view = pd.read_csv(f"{path}/spotify_songs.csv")  # 根據實際檔名修改

---

## Testing 

---

# version 1 
### 轉換時還是用到pandas中pd.dataframe

In [41]:
def _attempt_convert_type(value_str):
    try:
        return int(value_str)
    except ValueError:
        try:
            return float(value_str)
        except ValueError:
            return value_str.strip()

def load_csv(filePath, separator=","):
    """
    utilize 'with open' to read data row by row 
    """
    data_dict = {}
    header = []

    try:
        with open(filePath, "r", encoding="utf-8") as f:
            header_line = f.readline().strip()
            header = header_line.split(separator)
            data_dict = {col_name: [] for col_name in header}

            for line in f:
                cleaned_line = line.strip()

                if not cleaned_line:
                    continue

                values = cleaned_line.split(separator)

                if len(values) == len(header):
                    for i, col_name in enumerate(header):
    
                        converted_value = _attempt_convert_type(values[i]) 
                        data_dict[col_name].append(converted_value)

                # elif len(values) > 1:
                #     print(f"Warning: Skipping line due to mismatch in column count: {cleaned_line}")

    
        if data_dict:
            return pd.DataFrame(data_dict)
        else:
            print("File loaded but no data rows were found.")
    
    except FileNotFoundError:
        print(f"File doesn't exist at {filePath}")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [42]:
data = load_csv(f"{path}/spotify_songs.csv", separator=separator)

In [43]:
print(data.head())

                 track_id                                         track_name  \
0  6f807x0ima9a1j3VPbc7VN  I Don't Care (with Justin Bieber) - Loud Luxur...   
1  0r7CVbZTWZgbTCYdfa2P31                    Memories - Dillon Francis Remix   
2  1z1Hg7Vb0AhHDiEmnDE79l                    All the Time - Don Diablo Remix   
3  75FpbthrwQmzHlBJLuGdC7                  Call You Mine - Keanu Silva Remix   
4  1e8PAfcKUYoKkxPhrHqw4x            Someone You Loved - Future Humans Remix   

       track_artist  track_popularity          track_album_id  \
0        Ed Sheeran                66  2oCs0DGTsRO98Gh5ZSl2Cx   
1          Maroon 5                67  63rPSO264uRjW1X5E6cWv6   
2      Zara Larsson                70  1HoSmj2eLcsrR0vE9gThr4   
3  The Chainsmokers                60  1nqYsOef1yKKuGOVchbsk6   
4     Lewis Capaldi                69  7m7vv9wlQ4i0LFuJiE2zsQ   

                                    track_album_name track_album_release_date  \
0  I Don't Care (with Justin Bieber) [Loud Luxu

---

# version 2 
### advancing CSV loader
### define own Dataframe

In [4]:
def _attempt_convert_type(value_str):
    """Automatically convert data types"""
    # Handle empty strings
    if not value_str or value_str.strip() == '':
        return None
    
    value_str = value_str.strip()
    
    # Handle quoted strings
    if value_str.startswith('"') and value_str.endswith('"'):
        return value_str[1:-1]
    
    # Try converting to integer
    try:
        return int(value_str)
    except ValueError:
        pass
    
    # Try converting to float
    try:
        return float(value_str)
    except ValueError:
        pass
    
    # Handle boolean values
    if value_str.lower() in ['true', 'false']:
        return value_str.lower() == 'true'
    
    # Return string
    return value_str


def load_csv_advanced(filePath, separator=",", quote_char='"'):
    """
    advancing CSV loader, adressing quoted fields in CSV
    (ex: "Smith, John",25,USA)
    """
    def parse_csv_line(line, separator=",", quote_char='"'):
        """Parse a CSV line, correctly handling quoted fields"""
        values = []
        current_value = ""
        in_quotes = False
        
        for char in line:
            if char == quote_char:
                in_quotes = not in_quotes
            elif char == separator and not in_quotes:
                values.append(current_value)
                current_value = ""
            else:
                current_value += char
        
        # Add the last value
        values.append(current_value)
        return values
    
    data_dict = {}
    header = []
    
    try:
        with open(filePath, "r", encoding="utf-8") as f:
            # Read header line
            header_line = f.readline().strip()
            header = parse_csv_line(header_line, separator, quote_char)
            header = [col.strip() for col in header]
            
            data_dict = {col_name: [] for col_name in header}
            
            # Read data lines
            for line_number, line in enumerate(f, start=2):
                cleaned_line = line.strip()
                if not cleaned_line:
                    continue
                
                values = parse_csv_line(cleaned_line, separator, quote_char)
                
                if len(values) == len(header):
                    for i, col_name in enumerate(header):
                        converted_value = _attempt_convert_type(values[i])
                        data_dict[col_name].append(converted_value)
                else:
                    print(f"Warning: Line {line_number} column mismatch")
        
        return data_dict
        
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
class DataFrame:
    def __init__(self, data_dict):
        """
        initialize DataFrame with a dictionary of columns
        data_dict: {'column1': [values], 'column2': [values]}
        """
        if not data_dict:
            raise ValueError("Cannot create DataFrame from empty data")
        
        self.data = data_dict
        self.columns = list(data_dict.keys())
        
        # Check that all columns have the same length
        lengths = [len(v) for v in data_dict.values()]
        if len(set(lengths)) > 1:
            raise ValueError("All columns must have the same length")
        
        self.row_count = lengths[0] if lengths else 0
    
    def __repr__(self):
        """Print DataFrame"""
        if self.row_count == 0:
            return "Empty DataFrame"
        
        # calculate column widths based on header and first 10 rows
        col_widths = {}
        for col in self.columns:
            max_width = len(col)
            for val in self.data[col][:10]:  # only check first 10 rows
                max_width = max(max_width, len(str(val)))
            col_widths[col] = min(max_width, 20)  # max 20 characters
        
        # build table
        result = []
        
        # header row
        header = " | ".join(col.ljust(col_widths[col]) for col in self.columns)
        result.append(header)
        result.append("-" * len(header))
        
        # data rows (only show first 10 rows)
        display_rows = min(10, self.row_count)
        for i in range(display_rows):
            row = []
            for col in self.columns:
                val = str(self.data[col][i])
                if len(val) > col_widths[col]:
                    val = val[:col_widths[col]-3] + "..."
                row.append(val.ljust(col_widths[col]))
            result.append(" | ".join(row))
        
        if self.row_count > 10:
            result.append(f"\n... {self.row_count - 10} more rows")
        
        result.append(f"\nShape: ({self.row_count} rows, {len(self.columns)} columns)")
        
        return "\n".join(result)
    
    def __getitem__(self, key):
        """
        supporting df['column'] and df[condition]
        """
        if isinstance(key, str):
            # return single column
            return self.data[key]
        elif isinstance(key, list) and all(isinstance(k, str) for k in key):
            # return multiple columns
            new_data = {col: self.data[col] for col in key}
            return DataFrame(new_data)
        elif isinstance(key, list) and all(isinstance(k, bool) for k in key):
            # boolean indexing (for filtering)
            if len(key) != self.row_count:
                raise ValueError("Boolean index length mismatch")
            
            new_data = {col: [] for col in self.columns}
            for i, keep in enumerate(key):
                if keep:
                    for col in self.columns:
                        new_data[col].append(self.data[col][i])
            
            return DataFrame(new_data)
        else:
            raise TypeError(f"Invalid indexing type: {type(key)}")
    
    def __len__(self):
        """return"""
        return self.row_count

# ==================== 1. Filtering ====================
    
    def filter(self, condition):
        """
        based on condition filter rows
        
        Parameters:
            condition: function or list of booleans
                - function: lambda row: row['Age'] > 18
                - list of booleans: [True, False, True, ...]
        
        Returns:
            DataFrame: filtered new DataFrame
        
        Examples:
            df.filter(lambda row: row['GNP'] > 100000)
            df.filter([True, False, True])
        """
        if callable(condition):
            # function condition
            keep_rows = []
            for i in range(self.row_count):
                row = {col: self.data[col][i] for col in self.columns}
                keep_rows.append(condition(row))
            return self[keep_rows]
        
        elif isinstance(condition, list) and all(isinstance(k, bool) for k in condition):
            # list of booleans
            return self[condition]
        
        else:
            raise TypeError("Condition must be a callable or list of booleans")
    
    # ==================== 2. Projection (Select) ====================
    
    def select(self, columns):
        """
        select specific columns from the DataFrame
        
        Parameters:
            columns: str or list
                - single column: 'Name'
                - multiple columns: ['Name', 'Age']
        
        Returns:
            DataFrame: new DataFrame containing only the selected columns
        
        Examples:
            df.select('Name')
            df.select(['Name', 'Age'])
        """
        if isinstance(columns, str):
            columns = [columns]
        
        return self[columns]
    
    # ==================== 3. GroupBy ====================
    
    def groupby(self, by):
        """
        group by one or more columns
        
        Parameters:
            by: str or list - column name(s) to group by
        
        Returns:
            GroupBy: GroupBy object
        
        Examples:
            df.groupby('Continent')
            df.groupby(['Continent', 'Country'])
        """
        if isinstance(by, str):
            by = [by]
        
        return GroupBy(self, by)
    
    # ==================== 4. Join ====================
    
    def join(self, other, left_on, right_on, how='inner'):
        """
        Join with another DataFrame
        
        Parameters:
            other: DataFrame - the other DataFrame to join with
            left_on: str - join key from the left DataFrame
            right_on: str - join key from the right DataFrame
            how: str - join type ('inner', 'left', 'right', 'outer')
        
        Returns:
            DataFrame: new DataFrame after join
        
        Examples:
            countries.join(languages, left_on='Code', right_on='CountryCode')
        """
        # Create an index for the right DataFrame based on the join key
        right_index = {}
        for i, val in enumerate(other.data[right_on]):
            if val not in right_index:
                right_index[val] = []
            right_index[val].append(i)
        
        # Initialize result dictionary
        result_data = {col: [] for col in self.columns}
        for col in other.columns:
            if col != right_on:  # Avoid duplicates
                result_data[col] = []
        
        matched_right_indices = set()
        
        # Iterate over the left DataFrame
        for i in range(self.row_count):
            left_key = self.data[left_on][i]
            
            if left_key in right_index:
                # Found matches
                for right_i in right_index[left_key]:
                    matched_right_indices.add(right_i)
                    
                    # Add left DataFrame data
                    for col in self.columns:
                        result_data[col].append(self.data[col][i])
                    
                    # Add right DataFrame data
                    for col in other.columns:
                        if col != right_on:
                            result_data[col].append(other.data[col][right_i])
            elif how in ['left', 'outer']:
                # Left join or full outer join: keep left table rows
                for col in self.columns:
                    result_data[col].append(self.data[col][i])
                for col in other.columns:
                    if col != right_on:
                        result_data[col].append(None)
        
        # Handle right join or full outer join
        if how in ['right', 'outer']:
            for right_i in range(other.row_count):
                if right_i not in matched_right_indices:
                    # Add left table nulls
                    for col in self.columns:
                        result_data[col].append(None)
                    # Add right DataFrame data
                    for col in other.columns:
                        if col != right_on:
                            result_data[col].append(other.data[col][right_i])
        
        return DataFrame(result_data)
    
    # ==================== Helper Methods ====================
    
    def head(self, n=5):
        """ return the first n rows of the DataFrame """
        new_data = {col: self.data[col][:n] for col in self.columns}
        return DataFrame(new_data)
    
    def tail(self, n=5):
        """ return the last n rows of the DataFrame """
        new_data = {col: self.data[col][-n:] for col in self.columns}
        return DataFrame(new_data)
    
    def shape(self):
        """ return (number of rows, number of columns) """
        return (self.row_count, len(self.columns))
    
    def info(self):
        """ display DataFrame information """
        print(f"DataFrame Info:")
        print(f"Rows: {self.row_count}")
        print(f"Columns: {len(self.columns)}")
        print(f"\nColumn Names and Types:")
        for col in self.columns:
            sample_val = self.data[col][0] if self.row_count > 0 else None
            val_type = type(sample_val).__name__
            print(f"  {col}: {val_type}")
    
    @classmethod
    def from_csv(cls, filepath, separator=","):
        """ create DataFrame from CSV file """
        data_dict = load_csv(filepath, separator)
        if data_dict is None:
            raise ValueError(f"Failed to load CSV from {filepath}")
        return cls(data_dict)


# ==================== GroupBy Class ====================

class GroupBy:
    def __init__(self, dataframe, by):
        """
        GroupBy object
        
        Parameters:
            dataframe: DataFrame
            by: list - columns to group by
        """
        self.df = dataframe
        self.by = by
        self.groups = self._create_groups()
    
    def _create_groups(self):
        """create group indices"""
        groups = {}
        
        for i in range(self.df.row_count):
            # create group key
            key_values = tuple(self.df.data[col][i] for col in self.by)
            
            if key_values not in groups:
                groups[key_values] = []
            groups[key_values].append(i)
        
        return groups
    
    def aggregate(self, agg_dict):
        """
        Aggregate operation
        
        Parameters:
            agg_dict: dict - {column_name: aggregation_function}
                Supported functions: 'sum', 'mean', 'max', 'min', 'count', 'std'
        
        Returns:
            DataFrame: aggregation result
        
        Example:
            df.groupby('Continent').aggregate({'GNP': 'max', 'Population': 'sum'})
        """
        result_data = {col: [] for col in self.by}
        
        # Create result columns for each aggregation column
        for col, func in agg_dict.items():
            result_data[f"{col}_{func}"] = []
        
        # Aggregate each group
        for key_values, indices in self.groups.items():
            # Add group keys
            for i, col in enumerate(self.by):
                result_data[col].append(key_values[i])
            
            # Calculate each aggregation column
            for col, func_name in agg_dict.items():
                values = [self.df.data[col][i] for i in indices]
                # Filter out None values
                values = [v for v in values if v is not None]
                
                if not values:
                    result = None
                else:
                    result = self._apply_aggregation(values, func_name)
                
                result_data[f"{col}_{func_name}"].append(result)
        
        return DataFrame(result_data)
    
    def _apply_aggregation(self, values, func_name):
        """Apply aggregation function"""
        if func_name == 'sum':
            return sum(values)
        elif func_name == 'mean':
            return sum(values) / len(values)
        elif func_name == 'max':
            return max(values)
        elif func_name == 'min':
            return min(values)
        elif func_name == 'count':
            return len(values)
        elif func_name == 'std':
            mean = sum(values) / len(values)
            variance = sum((x - mean) ** 2 for x in values) / len(values)
            return variance ** 0.5
        else:
            raise ValueError(f"Unknown aggregation function: {func_name}")
    
    def size(self):
        """Return the size of each group"""
        result_data = {col: [] for col in self.by}
        result_data['size'] = []
        
        for key_values, indices in self.groups.items():
                    for i, col in enumerate(self.by):
                        result_data[col].append(key_values[i])
                    result_data['size'].append(len(indices))

        
        return DataFrame(result_data)

In [ ]:
# test code
if __name__ == "__main__":
    # Create test CSV file
    test_csv = """Name,Age,Country,GNP
USA,250,United States,21000000
China,70,People's Republic of China,14000000
Japan,150,Japan,5000000
Germany,100,Germany,4000000"""
    
    with open("test_countries.csv", "w") as f:
        f.write(test_csv)
    
    # Method 1: Directly read as dictionary
    data = load_csv("test_countries.csv")
    print("Raw data dict:")
    print(data)
    print()
    
    # Method 2: Using DataFrame class
    df = DataFrame.from_csv("test_countries.csv")
    print("DataFrame:")
    print(df)
    print()
    
    # Test column access
    print("Names column:")
    print(df['Name'])
    print()
    
    # Test multiple column selection
    print("Select Name and GNP:")
    print(df[['Name', 'GNP']])
    print()
    
    # Test filtering (you need to implement condition checking)
    print("Countries with GNP > 5000000:")
    condition = [gnp > 5000000 for gnp in df['GNP']]
    print(df[condition])

In [ ]:
# 1. loading data
countries = DataFrame.from_csv("countries.csv")
languages = DataFrame.from_csv("languages.csv")

# 2. filtering: find countries with GNP > 5 million
rich_countries = countries.filter(lambda row: row['GNP'] > 5000000)

# 3. selecting columns
result = rich_countries.select(['Name', 'GNP'])

# 4. grouping and aggregation: max GNP for each continent
continent_max = countries.groupby('Continent').aggregate({'GNP': 'max'})

# 5. joining: countries and languages
country_lang = countries.join(languages, 
                              left_on='Code', 
                              right_on='CountryCode')

# 6. chaining operations
result = (countries
    .filter(lambda row: row['Continent'] == 'Asia')
    .select(['Name', 'GNP'])
    .head(5))

In [31]:
df = load_csv_advanced("../data/spotify_songs.csv", separator=",")
df_dataframe = DataFrame(df)
df_dataframe.filter(lambda row: row['key'] > 5).head()

track_id             | track_name           | track_artist     | track_popularity | track_album_id       | track_album_name     | track_album_release_date | playlist_name | playlist_id          | playlist_genre | playlist_subgenre | danceability | energy | key | loudness | mode | speechiness | acousticness | instrumentalness | liveness | valence | tempo   | duration_ms
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
6f807x0ima9a1j3VP... | I Don't Care (wit... | Ed Sheeran       | 66               | 2oCs0DGTsRO98Gh5Z... | I Don't Care (wit... | 2019-06-14           | Pop Remix     | 37i9dQZF1DXcZDD7c... | pop            | dance pop         | 0.748        | 0.916  | 6  

--- 